# 06 · Rank estimate
**Question:** from someone's recent games, what rank do they play like? Useful when the rank is hidden or the player is unranked.
**Label:** the badge shown on the last game of a 20-game window, on a 0..71 level scale (`fe.badge_to_index`, 6 levels per tier).
**Inputs:** the same profile features as the clusters. **Split by account** (`ev.group_split`), never by row.
**Report:** MAE in levels, share in the same tier, share within one tier, against the baseline of always guessing the median.
Careful with framing on the site: show it as "plays like Oracle" with a range, not as a fact.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from broadway_ml.flatten import load_tables, item_catalog
from broadway_ml import features as fe, evaluate as ev, export as ex
T = load_tables('../data/tables')
{k: len(v) for k, v in T.items()}

In [ ]:
hist = pd.read_parquet('../data/tables/histories.parquet')
rd = fe.rank_dataset(hist, window=20, step=5)
tr, te = ev.group_split(rd.account_id, test_frac=0.2)
F = fe.PROFILE_COLS
print(len(rd), 'windows from', rd.account_id.nunique(), 'players'); rd.badge.value_counts().sort_index().plot.bar(figsize=(14, 3));

In [ ]:
med = np.median(rd.loc[tr, 'badge_idx'])
pd.DataFrame([ev.rank_report(rd.loc[te, 'badge_idx'], np.full(te.sum(), med), 'median')])

In [ ]:
# TODO(V): the model. Ridge is the exportable starting point; try ordinal regression or gradient boosting to see the ceiling
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
Z = StandardScaler().fit(rd.loc[tr, F])
model = Ridge(alpha=1.0).fit(Z.transform(rd.loc[tr, F]), rd.loc[tr, 'badge_idx'])
pred = model.predict(Z.transform(rd.loc[te, F]))
report = pd.DataFrame([ev.rank_report(rd.loc[te, 'badge_idx'], np.full(te.sum(), med), 'median'),
                       ev.rank_report(rd.loc[te, 'badge_idx'], pred, 'ridge')]).set_index('model'); report

In [ ]:
plt.figure(figsize=(5, 5)); plt.scatter(rd.loc[te, 'badge_idx'], pred, s=3, alpha=.3); plt.plot([0, 71], [0, 71], c='grey')
plt.xlabel('true level'); plt.ylabel('predicted level');

In [ ]:
ex.export_linear(model, F, '../models/rank_estimate.json', scaler=Z, name='rank-estimate', metrics=report.loc['ridge'].to_dict(),
                 notes='predicts the 0..71 level index; convert with indexToBadge in ml/web/score.ts')